# Email Campaign Recommender System

A two-tower retrieval model that predicts which subscribers are most likely to open a given campaign.

**Architecture overview:**
- **Baseline (ALS):** Collaborative filtering using Alternating Least Squares on a subscriber × campaign interaction matrix.
- **Two-Tower model:** Combines ALS embeddings with sentence-transformer text embeddings,
trained with a retrieval loss (in-batch negatives) using two-phase training.
Phase 1 warms up the text projection and fusion layers against frozen ALS embeddings.
Phase 2 fine-tunes all variables jointly, with ALS embeddings updated at a lower
learning rate to preserve their collaborative signal.

## 0. Install & Import

In [1]:
%pip install implicit
%pip install tf-keras -q
%pip install -q tensorflow-recommenders

In [11]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # required for tensorflow-recommenders compatibility

import tf_keras
import tensorflow as tf
tf.keras = tf_keras  # force tf.keras to point to Keras 2 before tfrs checks it

import tensorflow_recommenders as tfrs
import re
import json
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
import implicit as imp
from sentence_transformers import SentenceTransformer

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Load Data

In [15]:

DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/Small Dataset"

sends       = pd.read_csv(f"{DRIVE_PATH}/amplify_email_sends.csv", low_memory=False)
subscribers = pd.read_csv(f"{DRIVE_PATH}/amplify_email_subscribers.csv", low_memory=False)
subscribers = subscribers.rename(columns={"total emails sent": "total_emails_sent"})

#with open(f"{DRIVE_PATH}/amplify_email_campaigns.json") as f:
with open(f"{DRIVE_PATH}/amplify_email_campaigns.json") as f:
    campaigns = pd.DataFrame(json.load(f))

## 2. Clean & Prepare Data

In [16]:

# Drop sends that failed delivery or bounced — these carry no signal
sends_clean = sends[(sends["failed"] == 0) & (sends["bounced"] == 0)].copy()
print(f"Sends after filtering: {len(sends_clean):,} / {len(sends):,}")

# Parse timestamps and sort chronologically — required for the temporal split in Section 3
sends_clean["sent_at"] = pd.to_datetime(sends_clean["sent_at"], utc=True)
sends_clean = sends_clean.sort_values("sent_at").reset_index(drop=True)

Sends after filtering: 167,129 / 186,911


In [17]:
# Build a single text field per campaign by stripping HTML from email_html.
# (email_text is only present for ~40 campaigns, so we use email_html for all.)
def strip_html(html: str) -> str:
    if not isinstance(html, str):
        return ""
    text = re.sub(r"<style[^>]*>.*?</style>", " ", html, flags=re.DOTALL)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()[:2000]  # cap length

campaigns["campaign_text"] = (
    campaigns["subject"].fillna("") + " " + campaigns["email_html"].apply(strip_html)
).str.strip()

campaign_cols = campaigns[["id", "campaign_text"]].rename(columns={"id": "campaign_id"})

In [18]:
# Numeric feature columns used in the user tower
NUMERIC_FEATURES = [
    "total_emails_sent", "open_rate", "click_rate",
    "bounce_count", "unique_open_count", "unique_click_count",
    "active", "validated",
]

In [19]:
# Class imbalance summary (informational — used in the thesis write-up)
n_total   = len(sends_clean)
n_pos     = int(sends_clean["opened"].sum())
n_neg     = n_total - n_pos
n_clicked = int(sends_clean["clicked"].sum())

print("=== Implicit Feedback & Class Imbalance ===")
print(f"Total delivered sends : {n_total:>10,}")
print(f"Opened   (positive)  : {n_pos:>10,}  ({n_pos/n_total*100:.1f}%)")
print(f"Not opened (negative): {n_neg:>10,}  ({n_neg/n_total*100:.1f}%)")
print(f"Clicked              : {n_clicked:>10,}  ({n_clicked/n_total*100:.1f}%)")
print(f"\nImbalance ratio → 1 positive : {n_neg/n_pos:.2f} negatives")

=== Implicit Feedback & Class Imbalance ===
Total delivered sends :    167,129
Opened   (positive)  :     75,123  (44.9%)
Not opened (negative):     92,006  (55.1%)
Clicked              :      5,320  (3.2%)

Imbalance ratio → 1 positive : 1.22 negatives


## 3. Temporal Train / Test Split & Index Mappings

We split by time (last 20% of sends → test) rather than randomly.
This is more realistic: the model must predict future opens, not interpolate past ones.

In [20]:
TEST_FRAC  = 0.20  # last 20% of sends by time → test set
cutoff_unix = np.percentile(sends_clean["sent_at_unix"], (1 - TEST_FRAC) * 100)
cutoff_dt   = pd.to_datetime(cutoff_unix, unit="s", utc=True)

sends_train = sends_clean[sends_clean["sent_at_unix"] <  cutoff_unix].copy()
sends_test  = sends_clean[sends_clean["sent_at_unix"] >= cutoff_unix].copy()

print(f"Cutoff : {cutoff_dt.date()}")
print(f"Train  : {len(sends_train):,} sends "
      f"({sends_train['sent_at'].min().date()} → {sends_train['sent_at'].max().date()})")
print(f"Test   : {len(sends_test):,} sends "
      f"({sends_test['sent_at'].min().date()} → {sends_test['sent_at'].max().date()})")

Cutoff : 2025-05-27
Train  : 133,568 sends (2021-08-16 → 2025-05-27)
Test   : 33,561 sends (2025-05-27 → 2026-01-14)


In [21]:
subscriber_stats_train = (
    sends_train
    .groupby("subscriber_id")
    .agg(
        total_emails_sent = ("id",       "count"),
        open_rate         = ("opened",   "mean"),
        click_rate        = ("clicked",  "mean"),
        bounce_count      = ("bounced",  "sum"),
        unique_open_count = ("opened",   "sum"),
        unique_click_count= ("clicked",  "sum"),
    )
    .reset_index()
)

# Merge in static flags from the subscribers table
static_flags = subscribers[["id", "active", "validated"]].rename(
    columns={"id": "subscriber_id"}
)
subscriber_cols = subscriber_stats_train.merge(
    static_flags, on="subscriber_id", how="left"
)
subscriber_cols[["active", "validated"]] = (
    subscriber_cols[["active", "validated"]].fillna(0).astype("float32")
)

# Join sends → campaigns → subscribers into one flat dataframe
df = (
    sends_clean
    .merge(campaign_cols, on="campaign_id", how="inner")
    .merge(subscriber_cols, on="subscriber_id", how="inner")
)

print(f"Rows after joins: {len(df):,}")
print(f"Campaigns matched: {df['campaign_id'].nunique():,}")
print(f"Subscribers matched: {df['subscriber_id'].nunique():,}")

Rows after joins: 159,061
Campaigns matched: 715
Subscribers matched: 3,843


In [22]:
# Build each split independently so no test-time features contaminate training
def build_df(sends_split):
    return (
        sends_split
        .merge(campaign_cols, on="campaign_id", how="inner")
        .merge(subscriber_cols, on="subscriber_id", how="inner")
    )

data_train = build_df(sends_train)
data_test  = build_df(sends_test)
data_train[NUMERIC_FEATURES] = data_train[NUMERIC_FEATURES].fillna(0).astype("float32")

print(f"data_train: {len(data_train):,} rows")
print(f"data_test : {len(data_test):,} rows")

data_train: 133,568 rows
data_test : 25,493 rows


In [23]:
# Integer index mappings built from training set only.
# Subscribers or campaigns that appear only in the test set are cold-start entities.
sample_uids      = data_train["subscriber_id"].unique()
unique_campaigns = data_train["campaign_id"].unique()

user_to_index     = {uid: i for i, uid in enumerate(sample_uids)}
campaign_to_index = {cid: i for i, cid in enumerate(unique_campaigns)}
index_to_user     = {i: uid for uid, i in user_to_index.items()}
index_to_campaign = {i: cid for cid, i in campaign_to_index.items()}

# How much of the test set is cold-start?
pct_cold_users = (~data_test["subscriber_id"].isin(user_to_index)).sum() / len(data_test) * 100
pct_cold_camps = (~data_test["campaign_id"].isin(campaign_to_index)).sum() / len(data_test) * 100
print(f"Cold-start users in test     : {pct_cold_users:.1f}%")
print(f"Cold-start campaigns in test : {pct_cold_camps:.1f}%")

Cold-start users in test     : 0.0%
Cold-start campaigns in test : 99.2%


## 4. Build Interaction Pairs

We weight interactions by engagement strength:
- Delivered but not opened → weight 0 (explicit negative)
- Opened → weight 3
- Clicked → weight 8 (opened + clicked bonus)
- Converted → weight 18 (opened + clicked + converted bonus)

When a subscriber appears in both positives and negatives, we keep the higher weight.

In [24]:
ENGAGEMENT_WEIGHTS = {"opened": 2, "clicked": 5, "converted": 10}

# Positive pairs: sends that were opened
positives = sends_train[sends_train["opened"] == 1][
    ["subscriber_id", "campaign_id", "opened", "clicked", "converted"]
].copy()
positives["interaction_weight"] = (
    1
    + positives["opened"]    * ENGAGEMENT_WEIGHTS["opened"]
    + positives["clicked"]   * ENGAGEMENT_WEIGHTS["clicked"]
    + positives["converted"] * ENGAGEMENT_WEIGHTS["converted"]
)
positives["label"] = 1

# Negative pairs: delivered but never opened
negatives = sends_train[sends_train["opened"] == 0][["subscriber_id", "campaign_id"]].copy()
negatives["interaction_weight"] = 0.0
negatives["label"] = 0

# Merge and deduplicate: if a subscriber opened at least once, treat as positive
interactions_full = pd.concat([positives, negatives], ignore_index=True)
interactions_full = (
    interactions_full
    .groupby(["subscriber_id", "campaign_id"])
    .agg(interaction_weight=("interaction_weight", "max"),
         label=("label", "max"))
    .reset_index()
)

# Secondary filter: handles any subscribers/campaigns dropped during the inner joins
train_mask = (
    interactions_full["subscriber_id"].isin(user_to_index) &
    interactions_full["campaign_id"].isin(campaign_to_index)
)
grouped_interactions = (
    interactions_full[train_mask]
    .merge(subscriber_cols, on="subscriber_id", how="left")
)
grouped_interactions[NUMERIC_FEATURES] = (
    grouped_interactions[NUMERIC_FEATURES].fillna(0).astype("float32")
)

print(f"Positive pairs    : {(grouped_interactions['label']==1).sum():,}")
print(f"Negative pairs    : {(grouped_interactions['label']==0).sum():,}")
print(f"Unique subscribers: {len(sample_uids):,}")
print(f"Unique campaigns  : {len(unique_campaigns):,}")

Positive pairs    : 59,984
Negative pairs    : 73,584
Unique subscribers: 3,843
Unique campaigns  : 602


## 5. Train ALS Baseline
ALS (Alternating Least Squares) factorises the sparse user×item matrix into two low-rank factor matrices.
These learned factors also serve as warm-start embeddings for the two-tower model.

In [25]:
# Build the sparse interaction matrix (subscribers × campaigns)
rows      = grouped_interactions["subscriber_id"].map(user_to_index)
cols      = grouped_interactions["campaign_id"].map(campaign_to_index)
data_vals = grouped_interactions["interaction_weight"]

user_item_matrix     = coo_matrix((data_vals, (rows, cols)), shape=(len(sample_uids), len(unique_campaigns)))
user_item_matrix_csr = user_item_matrix.tocsr()  # CSR format required by implicit library

als_model = imp.als.AlternatingLeastSquares(factors=50, regularization=0.01, iterations=15)
als_model.fit(user_item_matrix_csr)

user_factors = als_model.user_factors  # shape: (n_subscribers, 50)
item_factors = als_model.item_factors  # shape: (n_campaigns,   50)
print("ALS trained successfully!")

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

ALS trained successfully!


## 6. Encode Campaign Text
We use a pretrained sentence transformer to turn each campaign's subject + body into a dense vector.
These text embeddings are used in the item tower to represent campaign *content*, complementing the
collaborative signal from ALS.

In [26]:
# One row per unique campaign, aligned to the ALS item index order
campaigns_unique = (
    data_train[["campaign_id", "campaign_text"]]
    .drop_duplicates(subset="campaign_id")
    .set_index("campaign_id")
    .loc[unique_campaigns]  # enforce same order as ALS item factors
    .reset_index()
)

model_text = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    cache_folder="/content/drive/MyDrive/models"
)
# normalize_embeddings=True → unit vectors, so cosine similarity = dot product
item_text_embeddings = model_text.encode(
    campaigns_unique["campaign_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)
print(f"Text embeddings shape: {item_text_embeddings.shape}")  # expect (n_campaigns, 384)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Text embeddings shape: (602, 384)


## 7. Define Towers

**User tower:** ALS user embedding (warm-started) + normalised subscriber stats → fused vector

**Item tower:** ALS item embedding (warm-started) + text embedding (projected) → fused vector

Both towers output L2-normalised vectors of dimension 50.

The `stats_normalizer` ensures features on very different scales (e.g. `open_rate` 0–1 vs
`unique_open_count` 0–5000+) are on the same scale before entering the network.

In [27]:
num_users = len(user_to_index)
num_items = len(campaign_to_index)

# --- Embedding layers initialised with ALS factors (trainable=True: fine-tuned at low LR) ---
user_embedding_layer = tf.keras.layers.Embedding(
    input_dim=num_users,
    output_dim=user_factors.shape[1],
    weights=[tf.constant(user_factors, dtype=tf.float32)],
    trainable=True
)
item_cf_embedding_layer = tf.keras.layers.Embedding(
    input_dim=num_items,
    output_dim=item_factors.shape[1],
    weights=[tf.constant(item_factors, dtype=tf.float32)],
    trainable=True
)

# --- Item tower layers ---
# Projects 384-dim text embedding directly to 50-dim (same as ALS factors)
text_projection = tf.keras.layers.Dense(
    units=item_factors.shape[1], name="text_direct_projection"
)
fusion_layer = tf.keras.layers.Dense(units=item_factors.shape[1], activation="relu")

# --- User tower layers ---
# Normalise subscriber stats so all features are on the same scale
stats_normalizer = tf.keras.layers.Normalization(axis=-1)
stats_normalizer.adapt(grouped_interactions[NUMERIC_FEATURES].values.astype("float32"))

subscriber_stats_layer = tf.keras.layers.Dense(units=32, activation="relu")
user_fusion_layer      = tf.keras.layers.Dense(units=user_factors.shape[1], activation="relu")


def user_model(user_idx, subscriber_stats):
    """Returns L2-normalised user embedding (batch_size, 50)."""
    cf_emb     = user_embedding_layer(user_idx)
    stats_proj = subscriber_stats_layer(stats_normalizer(subscriber_stats))
    fused      = user_fusion_layer(tf.concat([cf_emb, stats_proj], axis=-1))
    return tf.nn.l2_normalize(fused, axis=1)

def item_model(item_idx, text_emb):
    """Returns L2-normalised item embedding (batch_size, 50)."""
    cf_emb    = item_cf_embedding_layer(item_idx)
    text_proj = text_projection(text_emb)
    fused_emb = fusion_layer(tf.concat([cf_emb, text_proj], axis=-1))
    return tf.nn.l2_normalize(fused_emb, axis=1)

## 8. Build tf.data Pipelines

In [28]:
def map_interaction_to_tensors(x):
    """Maps raw IDs to integer indices and looks up the precomputed text embedding."""
    user_idx = tf.py_function(
        lambda uid: user_to_index[int(uid.numpy())], [x["subscriber_id"]], tf.int64)
    item_idx = tf.py_function(
        lambda cid: campaign_to_index[int(cid.numpy())], [x["campaign_id"]], tf.int64)
    text_emb = tf.py_function(
        lambda cid: item_text_embeddings[campaign_to_index[int(cid.numpy())]],
        [x["campaign_id"]], tf.float32)

    user_idx.set_shape([])
    item_idx.set_shape([])
    text_emb.set_shape(item_text_embeddings.shape[1:])

    return {
        "user_idx":         user_idx,
        "item_idx":         item_idx,
        "text_emb":         text_emb,
        "subscriber_stats": x["subscriber_stats"],
        "label":            tf.cast(x["label"], tf.int32),
    }


def map_candidate_to_tensors(x):
    """Candidate pool — items only, no user info needed."""
    item_idx = tf.py_function(
        lambda cid: campaign_to_index[int(cid.numpy())], [x["campaign_id"]], tf.int64)
    text_emb = tf.py_function(
        lambda cid: item_text_embeddings[campaign_to_index[int(cid.numpy())]],
        [x["campaign_id"]], tf.float32)

    item_idx.set_shape([])
    text_emb.set_shape(item_text_embeddings.shape[1:])
    return {"item_idx": item_idx, "text_emb": text_emb}


full_ds = (
    tf.data.Dataset.from_tensor_slices({
        "subscriber_id":    grouped_interactions["subscriber_id"].values,
        "campaign_id":      grouped_interactions["campaign_id"].values,
        "label":            grouped_interactions["label"].values.astype("int32"),
        "subscriber_stats": tf.constant(grouped_interactions[NUMERIC_FEATURES].values, dtype=tf.float32),
    })
    .map(map_interaction_to_tensors, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(len(grouped_interactions), seed=42)
)

train_ds = full_ds.batch(512).prefetch(tf.data.AUTOTUNE)

# Candidate pool: all unique training campaigns — used to compute retrieval metrics
candidate_ds = (
    tf.data.Dataset.from_tensor_slices({"campaign_id": campaigns_unique["campaign_id"].values})
    .map(map_candidate_to_tensors, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
)

In [29]:
# Total subscribers in database
print(f"Total subscribers in database: {len(subscribers):,}")

# Subscribers who received at least one send (any period)
received_any = sends_clean["subscriber_id"].nunique()
print(f"Received at least one send (full dataset): {received_any:,}")

# Subscribers who received at least one send in training period
received_train = sends_train["subscriber_id"].nunique()
print(f"Received at least one send in training: {received_train:,}")

# Subscribers in grouped_interactions (training candidates)
print(f"In training interaction matrix: {len(sample_uids):,}")

# Never received any email
never_received = len(subscribers) - received_any
print(f"Never received any email: {never_received:,}")

Total subscribers in database: 6,624
Received at least one send (full dataset): 4,681
Received at least one send in training: 3,843
In training interaction matrix: 3,843
Never received any email: 1,943


## 9. Define & Train Two-Tower Model

### Loss function
- **Retrieval loss** (in-batch negatives): applied only to *positive* pairs. Other items in the same batch are treated as negatives. `temperature=0.1` prevents softmax denominator explosion at small batch sizes.

### Two-phase training
- **Phase 1:** New layers only, retrieval loss only. Warms up the text projection and fusion layers before the ALS embeddings are unlocked.
- **Phase 2:** All variables. ALS embeddings use a lower learning-rate (0.003) to preserve their learned signal; new layers use 0.07

In [30]:
class TwoTowerRetrievalModel(tfrs.models.Model):
    def __init__(self, user_model_fn, item_model_fn, item_candidates):
        super().__init__()
        self.user_model_fn = user_model_fn
        self.item_model_fn = item_model_fn
        self.retrieval_task = tfrs.tasks.Retrieval(
            temperature=0.1,
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=item_candidates.batch(128).map(
                    lambda x: item_model_fn(x["item_idx"], x["text_emb"])
                )
            )
        )

    def compute_loss(self, features, training=False):
        user_emb = self.user_model_fn(features["user_idx"], features["subscriber_stats"])
        item_emb = self.item_model_fn(features["item_idx"], features["text_emb"])
        pos_mask = tf.cast(features["label"], tf.bool)
        return tf.cond(
            tf.reduce_any(pos_mask),
            lambda: self.retrieval_task(
                tf.boolean_mask(user_emb, pos_mask),
                tf.boolean_mask(item_emb, pos_mask)
            ),
            lambda: 0.0
        )


retrieval_model = TwoTowerRetrievalModel(user_model, item_model, candidate_ds)

# Split variables into two groups for the dual-optimiser strategy
embedding_vars = (
    user_embedding_layer.trainable_variables +
    item_cf_embedding_layer.trainable_variables
)
new_layer_vars = (
    text_projection.trainable_variables +
    fusion_layer.trainable_variables +
    subscriber_stats_layer.trainable_variables +
    user_fusion_layer.trainable_variables
)

embedding_optimizer = tf.keras.optimizers.Adagrad(0.003)  # slow: preserve ALS signal
optimizer           = tf.keras.optimizers.Adagrad(0.07)   # fast: train new layers from scratch

In [31]:
# ── Phase 1: Retrieval loss — new layers only ─────────────────────────────────
# Warms up text projection and fusion layers against frozen ALS embeddings.
# Prevents retrieval gradients from thrashing warm-started ALS factors on epoch 1.

@tf.function
def train_step_phase1(features):
    pos_mask = tf.cast(features["label"], tf.bool)
    with tf.GradientTape() as tape:
        user_emb = user_model(
            tf.boolean_mask(features["user_idx"], pos_mask),
            tf.boolean_mask(features["subscriber_stats"], pos_mask)
        )
        item_emb = item_model(
            tf.boolean_mask(features["item_idx"], pos_mask),
            tf.boolean_mask(features["text_emb"], pos_mask)
        )
        loss = retrieval_model.retrieval_task(
            user_emb, item_emb, compute_metrics=False
        )
    grads = tape.gradient(loss, new_layer_vars)
    optimizer.apply_gradients(zip(grads, new_layer_vars))
    return loss


# ── Phase 2: Retrieval loss — all variables ───────────────────────────────────
# Towers are now stable. ALS embeddings unlocked at low LR to refine
# collaborative signal without overwriting the warm-start.

@tf.function
def train_step(features):
    pos_mask = tf.cast(features["label"], tf.bool)
    with tf.GradientTape() as tape:
        user_emb = user_model(
            tf.boolean_mask(features["user_idx"], pos_mask),
            tf.boolean_mask(features["subscriber_stats"], pos_mask)
        )
        item_emb = item_model(
            tf.boolean_mask(features["item_idx"], pos_mask),
            tf.boolean_mask(features["text_emb"], pos_mask)
        )
        loss = retrieval_model.retrieval_task(user_emb, item_emb)
    all_vars = embedding_vars + new_layer_vars
    grads = tape.gradient(loss, all_vars)
    embedding_optimizer.apply_gradients(zip(grads[:len(embedding_vars)], embedding_vars))
    optimizer.apply_gradients(zip(grads[len(embedding_vars):], new_layer_vars))
    return loss

In [32]:
PHASE1_EPOCHS = 8
PHASE2_EPOCHS = 6

print("=== PHASE 1: Retrieval loss — new layers only ===")
for epoch in range(PHASE1_EPOCHS):
    total, batches = 0.0, 0
    for batch in train_ds:
        total += train_step_phase1(batch)
        batches += 1
    print(f" P1 Epoch {epoch+1:02d}: train={total/batches:.4f}")

print("=== PHASE 2: Retrieval loss — all variables ===")
for epoch in range(PHASE2_EPOCHS):
    total, batches = 0.0, 0
    for batch in train_ds:
        total += train_step(batch)
        batches += 1
    print(f" P2 Epoch {epoch+1:02d}: train={total/batches:.4f}")

=== PHASE 1: Retrieval loss — new layers only ===
 P1 Epoch 01: train=1084.4766
 P1 Epoch 02: train=1034.2908
 P1 Epoch 03: train=1026.0840
 P1 Epoch 04: train=1022.5773
 P1 Epoch 05: train=1020.7479
 P1 Epoch 06: train=1019.6779
 P1 Epoch 07: train=1018.5923
 P1 Epoch 08: train=1017.8740
=== PHASE 2: Retrieval loss — all variables ===
 P2 Epoch 01: train=1023.7126
 P2 Epoch 02: train=1017.7490
 P2 Epoch 03: train=1016.4955
 P2 Epoch 04: train=1015.4701
 P2 Epoch 05: train=1014.5940
 P2 Epoch 06: train=1014.1098


## 10. Evaluate — Recall@K and NDCG@K

**Recall@K**: of the subscribers who actually opened a campaign, what fraction appear in the top-K predictions?

**NDCG@K**: like Recall@K, but rewards putting correct predictions *higher* in the ranking.

We evaluate two scenarios:
1. Cold-start evaluation — campaigns the model has never seen
   (99.4% of test campaigns; the primary deployment scenario)
2. Popularity baseline — naive floor every model must beat

In [33]:
# Build a (num_users, 50) matrix of all user embeddings for fast batch scoring
subscriber_stats_matrix = (
    pd.DataFrame({"subscriber_id": list(index_to_user.values()), "user_idx": list(index_to_user.keys())})
    .merge(subscriber_cols, on="subscriber_id", how="left")
    .sort_values("user_idx")[NUMERIC_FEATURES]
    .fillna(0).astype("float32").values
)

all_user_embeddings = user_model(
    tf.range(num_users),
    tf.constant(subscriber_stats_matrix, dtype=tf.float32)
)

K = 100

def recall_ndcg_at_k(top_k_indices, true_uid_set, k):
    top_k_uids = {index_to_user[i] for i in top_k_indices}
    hits       = len(true_uid_set & top_k_uids)
    recall     = hits / min(len(true_uid_set), k)
    dcg        = sum(1/np.log2(r+2) for r, i in enumerate(top_k_indices)
                     if index_to_user[i] in true_uid_set)
    idcg       = sum(1/np.log2(i+2) for i in range(min(len(true_uid_set), k)))
    return recall, (dcg / idcg if idcg > 0 else 0.0)

In [34]:
# ── Cold-start campaign evaluation ────────────────────────────────────────────
# Tests exclusively campaigns the model has never seen during training.
# ALS has no item factor for these → uses mean factor (effectively random).
# Two-tower relies on text embeddings → real test of content generalisation.

cold_campaign_ids = [cid for cid in data_test["campaign_id"].unique() if cid not in campaign_to_index]
print(f"Cold-start campaigns (unseen): {len(cold_campaign_ids)} / {data_test['campaign_id'].nunique()}")

# Encode cold campaign text
cold_campaigns_df = (
    campaigns[campaigns["id"].isin(cold_campaign_ids)][["id", "campaign_text"]]
    .rename(columns={"id": "campaign_id"})
    .drop_duplicates(subset="campaign_id").reset_index(drop=True)
)
cold_text_embeddings = model_text.encode(
    cold_campaigns_df["campaign_text"].tolist(),
    normalize_embeddings=True, show_progress_bar=True, batch_size=64
)
cold_text_emb_lookup = {
    row["campaign_id"]: cold_text_embeddings[i] for i, row in cold_campaigns_df.iterrows()
}

# Ground truth: cold campaigns with at least one known opener in test set
cold_ground_truth = (
    data_test[
        data_test["campaign_id"].isin(cold_campaign_ids) &
        data_test["subscriber_id"].isin(user_to_index) &
        (data_test["opened"] == 1)
    ]
    .groupby("campaign_id")["subscriber_id"].apply(list).to_dict()
)
print(f"Cold campaigns with ≥1 test opener: {len(cold_ground_truth)}")

# Build cold-start item embeddings directly (no weight patching needed):
# fuse mean ALS factor with the campaign's actual text embedding
mean_cf          = tf.constant(item_factors.mean(axis=0, keepdims=True), dtype=tf.float32)
mean_item_factor = item_factors.mean(axis=0)  # numpy version for ALS scoring

cold_als_recalls, cold_als_ndcgs = [], []
cold_tt_recalls,  cold_tt_ndcgs  = [], []

for campaign_id, true_uids in cold_ground_truth.items():
    true_uid_set = set(true_uids)

    # ALS fallback: mean factor dotted against all user factors
    als_top_k = np.argsort(user_factors @ mean_item_factor)[::-1][:K]
    r, n = recall_ndcg_at_k(als_top_k, true_uid_set, K)
    cold_als_recalls.append(r); cold_als_ndcgs.append(n)

    # Two-Tower: build item embedding from mean CF + text (no embedding layer patching)
    text_vec = cold_text_emb_lookup.get(campaign_id)
    if text_vec is None:
        continue
    text_proj = text_projection(
        tf.expand_dims(tf.constant(text_vec, dtype=tf.float32), 0)
    )
    fused    = fusion_layer(tf.concat([mean_cf, text_proj], axis=-1))
    item_emb = tf.nn.l2_normalize(fused, axis=1)
    scores   = tf.linalg.matmul(item_emb, all_user_embeddings, transpose_b=True)[0].numpy()
    r, n = recall_ndcg_at_k(np.argsort(scores)[::-1][:K], true_uid_set, K)
    cold_tt_recalls.append(r); cold_tt_ndcgs.append(n)


# ── Popularity baseline: always recommend globally most active subscribers ────
popularity_top_k = (
    grouped_interactions[grouped_interactions["label"] == 1]
    .groupby("subscriber_id")["interaction_weight"].sum()
    .reset_index()
    .assign(user_idx=lambda d: d["subscriber_id"].map(user_to_index))
    .drop_duplicates(subset=["user_idx"])
    .sort_values("interaction_weight", ascending=False)
    ["user_idx"].astype(int).values
)

cold_pop_recalls = [recall_ndcg_at_k(popularity_top_k[:K], set(u), K)[0]
                    for u in cold_ground_truth.values()]
cold_pop_ndcgs   = [recall_ndcg_at_k(popularity_top_k[:K], set(u), K)[1]
                    for u in cold_ground_truth.values()]

# ── Print cold-start results ──────────────────────────────────────────────────
print(f"\n{'Model':<35} {'Recall@'+str(K):<15} {'NDCG@'+str(K)}")
print("─" * 60)
print(f"{'Popularity (global opens)':<35} {np.mean(cold_pop_recalls):.4f}         {np.mean(cold_pop_ndcgs):.4f}")
print(f"{'ALS (mean factor — no signal)':<35} {np.mean(cold_als_recalls):.4f}         {np.mean(cold_als_ndcgs):.4f}")
print(f"{'Two-Tower (text embedding only)':<35} {np.mean(cold_tt_recalls):.4f}         {np.mean(cold_tt_ndcgs):.4f}")

# ── Relative uplift vs popularity floor ───────────────────────────────────────
if np.mean(cold_pop_recalls) > 0:
    tt_vs_pop_r  = (np.mean(cold_tt_recalls)  - np.mean(cold_pop_recalls)) / np.mean(cold_pop_recalls) * 100
    tt_vs_pop_n  = (np.mean(cold_tt_ndcgs)    - np.mean(cold_pop_ndcgs))   / np.mean(cold_pop_ndcgs)   * 100
    als_vs_pop_r = (np.mean(cold_als_recalls)  - np.mean(cold_pop_recalls)) / np.mean(cold_pop_recalls) * 100
    als_vs_pop_n = (np.mean(cold_als_ndcgs)    - np.mean(cold_pop_ndcgs))   / np.mean(cold_pop_ndcgs)   * 100
    print(f"\nRelative uplift vs popularity floor:")
    print(f"  ALS       vs popularity → Recall: {als_vs_pop_r:+.1f}%  NDCG: {als_vs_pop_n:+.1f}%")
    print(f"  Two-Tower vs popularity → Recall: {tt_vs_pop_r:+.1f}%   NDCG: {tt_vs_pop_n:+.1f}%")

print(f"\nCold campaigns evaluated: {len(cold_tt_recalls)}")

Cold-start campaigns (unseen): 113 / 114


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Cold campaigns with ≥1 test opener: 107

Model                               Recall@100      NDCG@100
────────────────────────────────────────────────────────────
Popularity (global opens)           0.2160         0.1731
ALS (mean factor — no signal)       0.1389         0.1406
Two-Tower (text embedding only)     0.1439         0.1322

Relative uplift vs popularity floor:
  ALS       vs popularity → Recall: -35.7%  NDCG: -18.8%
  Two-Tower vs popularity → Recall: -33.4%   NDCG: -23.6%

Cold campaigns evaluated: 107


In [35]:
print(f"{'K':<6} {'Pop Recall':<14} {'Pop NDCG':<14} {'ALS Recall':<14} {'ALS NDCG':<14} {'TT Recall':<14} {'TT NDCG'}")
print("─" * 90)

for k in [10, 25, 50, 100, 200]:
    pop_r, pop_n = [], []
    als_r, als_n = [], []
    tt_r,  tt_n  = [], []

    for campaign_id, true_uids in cold_ground_truth.items():
        true_uid_set = set(true_uids)

        # Popularity
        top     = popularity_top_k[:k]
        r, n    = recall_ndcg_at_k(top, true_uid_set, k)
        pop_r.append(r); pop_n.append(n)

        # ALS
        scores  = user_factors @ mean_item_factor
        top     = np.argsort(scores)[::-1][:k]
        r, n    = recall_ndcg_at_k(top, true_uid_set, k)
        als_r.append(r); als_n.append(n)

        # Two-Tower
        text_vec = cold_text_emb_lookup.get(campaign_id)
        if text_vec is None: continue
        tp   = text_projection(tf.expand_dims(tf.constant(text_vec, dtype=tf.float32), 0))
        fu   = fusion_layer(tf.concat([mean_cf, tp], axis=-1))
        ie   = tf.nn.l2_normalize(fu, axis=1)
        s    = tf.linalg.matmul(ie, all_user_embeddings, transpose_b=True)[0].numpy()
        top  = np.argsort(s)[::-1][:k]
        r, n = recall_ndcg_at_k(top, true_uid_set, k)
        tt_r.append(r); tt_n.append(n)

    print(f"{k:<6} "
          f"{np.mean(pop_r):<14.4f} {np.mean(pop_n):<14.4f} "
          f"{np.mean(als_r):<14.4f} {np.mean(als_n):<14.4f} "
          f"{np.mean(tt_r):<14.4f} {np.mean(tt_n):.4f}")



K      Pop Recall     Pop NDCG       ALS Recall     ALS NDCG       TT Recall      TT NDCG
──────────────────────────────────────────────────────────────────────────────────────────
10     0.2199         0.2201         0.1942         0.2072         0.1037         0.1043
25     0.1949         0.1961         0.1684         0.1803         0.1239         0.1190
50     0.1675         0.1676         0.1456         0.1551         0.1358         0.1284
100    0.2160         0.1731         0.1389         0.1406         0.1439         0.1322
200    0.2579         0.1870         0.1586         0.1409         0.1984         0.1541


---

## 11. Multi-Seed Evaluation

Runs the full ALS → Two-Tower pipeline 5 times with different random seeds to measure
result variance.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]

all_run_results = []

for seed_idx, SEED in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"  SEED {SEED} (run {seed_idx+1}/{len(SEEDS)})")
    print(f"{'='*60}")

    # ── ALS ────────────────────────────────────────────────────────
    _als = imp.als.AlternatingLeastSquares(
        factors=50, regularization=0.01, iterations=15, random_state=SEED
    )
    _als.fit(user_item_matrix_csr)
    _user_factors = _als.user_factors
    _item_factors = _als.item_factors
    print(f"ALS trained with seed {SEED}")

    # ── Reinitialise all layers ────────────────────────────────────
    _user_emb_layer = tf.keras.layers.Embedding(
        input_dim=num_users, output_dim=_user_factors.shape[1],
        weights=[tf.constant(_user_factors, dtype=tf.float32)], trainable=True
    )
    _item_cf_layer = tf.keras.layers.Embedding(
        input_dim=num_items, output_dim=_item_factors.shape[1],
        weights=[tf.constant(_item_factors, dtype=tf.float32)], trainable=True
    )
    _text_proj     = tf.keras.layers.Dense(units=_item_factors.shape[1], name="text_direct_projection")
    _fusion        = tf.keras.layers.Dense(units=_item_factors.shape[1], activation="relu")
    _stats_norm    = tf.keras.layers.Normalization(axis=-1)
    _stats_norm.adapt(grouped_interactions[NUMERIC_FEATURES].values.astype("float32"))
    _stats_layer   = tf.keras.layers.Dense(units=32, activation="relu")
    _user_fusion   = tf.keras.layers.Dense(units=_user_factors.shape[1], activation="relu")

    def _user_model(user_idx, subscriber_stats):
        cf = _user_emb_layer(user_idx)
        sp = _stats_layer(_stats_norm(subscriber_stats))
        return tf.nn.l2_normalize(_user_fusion(tf.concat([cf, sp], axis=-1)), axis=1)

    def _item_model(item_idx, text_emb):
        cf = _item_cf_layer(item_idx)
        tp = _text_proj(text_emb)
        return tf.nn.l2_normalize(_fusion(tf.concat([cf, tp], axis=-1)), axis=1)

    # ── Data pipeline (same structure, fresh shuffle) ──────────────
    _full_ds = (
        tf.data.Dataset.from_tensor_slices({
            "subscriber_id":    grouped_interactions["subscriber_id"].values,
            "campaign_id":      grouped_interactions["campaign_id"].values,
            "label":            grouped_interactions["label"].values.astype("int32"),
            "subscriber_stats": tf.constant(grouped_interactions[NUMERIC_FEATURES].values, dtype=tf.float32),
        })
        .map(map_interaction_to_tensors, num_parallel_calls=tf.data.AUTOTUNE)
        .cache()
        .shuffle(len(grouped_interactions), seed=SEED)
    )
    _train_ds = _full_ds.batch(512).prefetch(tf.data.AUTOTUNE)

    _candidate_ds = (
        tf.data.Dataset.from_tensor_slices({"campaign_id": campaigns_unique["campaign_id"].values})
        .map(map_candidate_to_tensors, num_parallel_calls=tf.data.AUTOTUNE)
    )

    # ── Model & optimisers ─────────────────────────────────────────
    _model = TwoTowerRetrievalModel(_user_model, _item_model, _candidate_ds)

    _emb_vars = _user_emb_layer.trainable_variables + _item_cf_layer.trainable_variables
    _new_vars = (
        _text_proj.trainable_variables + _fusion.trainable_variables +
        _stats_layer.trainable_variables + _user_fusion.trainable_variables
    )
    _emb_opt = tf.keras.optimizers.Adagrad(0.003)
    _opt     = tf.keras.optimizers.Adagrad(0.07)

    # Phase 1
    def _train_p1(features):
        pm = tf.cast(features["label"], tf.bool)
        with tf.GradientTape() as tape:
            ue = _user_model(tf.boolean_mask(features["user_idx"], pm),
                             tf.boolean_mask(features["subscriber_stats"], pm))
            ie = _item_model(tf.boolean_mask(features["item_idx"], pm),
                             tf.boolean_mask(features["text_emb"], pm))
            loss = _model.retrieval_task(ue, ie, compute_metrics=False)
        grads = tape.gradient(loss, _new_vars)
        _opt.apply_gradients(zip(grads, _new_vars))
        return loss

    # Phase 2
    def _train_p2(features):
        pm = tf.cast(features["label"], tf.bool)
        with tf.GradientTape() as tape:
            ue = _user_model(tf.boolean_mask(features["user_idx"], pm),
                             tf.boolean_mask(features["subscriber_stats"], pm))
            ie = _item_model(tf.boolean_mask(features["item_idx"], pm),
                             tf.boolean_mask(features["text_emb"], pm))
            loss = _model.retrieval_task(ue, ie)
        av = _emb_vars + _new_vars
        grads = tape.gradient(loss, av)
        _emb_opt.apply_gradients(zip(grads[:len(_emb_vars)], _emb_vars))
        _opt.apply_gradients(zip(grads[len(_emb_vars):], _new_vars))
        return loss

    PHASE1_EPOCHS = 8
    PHASE2_EPOCHS = 6

    print("=== PHASE 1 ===")
    for ep in range(PHASE1_EPOCHS):
        total, batches = 0.0, 0
        for batch in _train_ds:
            total += _train_p1(batch)
            batches += 1
        print(f"  P1 Epoch {ep+1:02d}: train={total/batches:.4f}")

    print("=== PHASE 2 ===")
    for ep in range(PHASE2_EPOCHS):
        total, batches = 0.0, 0
        for batch in _train_ds:
            total += _train_p2(batch)
            batches += 1
        print(f"  P2 Epoch {ep+1:02d}: train={total/batches:.4f}")

    # ── Evaluate ───────────────────────────────────────────────────
    _all_user_emb = _user_model(
        tf.range(num_users),
        tf.constant(subscriber_stats_matrix, dtype=tf.float32)
    )

    _mean_cf = tf.constant(_item_factors.mean(axis=0, keepdims=True), dtype=tf.float32)
    _mean_if = _item_factors.mean(axis=0)

    # Precompute all scores once, then slice for each K
    _als_all_scores = {}
    _tt_all_scores  = {}

    for cid, uids in cold_ground_truth.items():
        _als_all_scores[cid] = np.argsort(_user_factors @ _mean_if)[::-1]

        tv = cold_text_emb_lookup.get(cid)
        if tv is None: continue
        tp = _text_proj(tf.expand_dims(tf.constant(tv, dtype=tf.float32), 0))
        fu = _fusion(tf.concat([_mean_cf, tp], axis=-1))
        ie = tf.nn.l2_normalize(fu, axis=1)
        s  = tf.linalg.matmul(ie, _all_user_emb, transpose_b=True)[0].numpy()
        _tt_all_scores[cid] = np.argsort(s)[::-1]

    K_VALUES = [10, 25, 50, 100, 200]

    seed_results = {"seed": SEED}
    for k in K_VALUES:
        _c_als_r, _c_als_n = [], []
        _c_tt_r,  _c_tt_n  = [], []
        _c_pop_r, _c_pop_n = [], []

        for cid, uids in cold_ground_truth.items():
            us = set(uids)

            r, n = recall_ndcg_at_k(_als_all_scores[cid][:k], us, k)
            _c_als_r.append(r); _c_als_n.append(n)

            r, n = recall_ndcg_at_k(popularity_top_k[:k], us, k)
            _c_pop_r.append(r); _c_pop_n.append(n)

            if cid in _tt_all_scores:
                r, n = recall_ndcg_at_k(_tt_all_scores[cid][:k], us, k)
                _c_tt_r.append(r); _c_tt_n.append(n)

        seed_results[f"pop_r@{k}"]  = np.mean(_c_pop_r)
        seed_results[f"als_r@{k}"]  = np.mean(_c_als_r)
        seed_results[f"tt_r@{k}"]   = np.mean(_c_tt_r)
        seed_results[f"pop_n@{k}"]  = np.mean(_c_pop_n)
        seed_results[f"als_n@{k}"]  = np.mean(_c_als_n)
        seed_results[f"tt_n@{k}"]   = np.mean(_c_tt_n)

    all_run_results.append(seed_results)

    # Print this seed's results
    print(f"\n{'K':<6} {'Pop R':<10} {'Pop N':<10} {'ALS R':<10} {'ALS N':<10} {'TT R':<10} {'TT N':<10}")
    print("─" * 66)
    for k in K_VALUES:
      print(f"{k:<6} "
          f"{seed_results[f'pop_r@{k}']:<10.4f} "
          f"{seed_results[f'pop_n@{k}']:<10.4f} "
          f"{seed_results[f'als_r@{k}']:<10.4f} "
          f"{seed_results[f'als_n@{k}']:<10.4f} "
          f"{seed_results[f'tt_r@{k}']:<10.4f} "
          f"{seed_results[f'tt_n@{k}']:.4f}")

In [ ]:
results_df = pd.DataFrame(all_run_results)

K_VALUES = [10, 25, 50, 100, 200]

print("=" * 75)
print("  AGGREGATED RECALL@K ACROSS 5 SEEDS (mean ± std)")
print("=" * 75)
print(f"\n{'K':<6} {'Pop Recall':<16} {'ALS Recall':<16} {'TT Recall':<16}")
print("─" * 55)
for k in K_VALUES:
    print(f"{k:<6} "
          f"{results_df[f'pop_r@{k}'].mean():.4f} ± {results_df[f'pop_r@{k}'].std():.4f}  "
          f"{results_df[f'als_r@{k}'].mean():.4f} ± {results_df[f'als_r@{k}'].std():.4f}  "
          f"{results_df[f'tt_r@{k}'].mean():.4f} ± {results_df[f'tt_r@{k}'].std():.4f}")

print(f"\n{'K':<6} {'Pop NDCG':<16} {'ALS NDCG':<16} {'TT NDCG':<16}")
print("─" * 55)
for k in K_VALUES:
    print(f"{k:<6} "
          f"{results_df[f'pop_n@{k}'].mean():.4f} ± {results_df[f'pop_n@{k}'].std():.4f}  "
          f"{results_df[f'als_n@{k}'].mean():.4f} ± {results_df[f'als_n@{k}'].std():.4f}  "
          f"{results_df[f'tt_n@{k}'].mean():.4f} ± {results_df[f'tt_n@{k}'].std():.4f}")
print(results_df.to_string(index=False))